In [2]:
!pip install -q sentence-transformers scikit-learn pandas skl2onnx onnxruntime tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 26.2 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Saving hinglish-clean-500.csv to hinglish-clean-500.csv
Saving hinglish-stealth-110-heldout.csv to hinglish-stealth-110-heldout.csv
Saving hinglish-stealth-250.csv to hinglish-stealth-250.csv
Saving srinivasan_dataset.xlsx to srinivasan_dataset.xlsx
Uploaded files: ['hinglish-clean-500.csv', 'hinglish-stealth-110-heldout.csv', 'hinglish-stealth-250.csv', 'srinivasan_dataset.xlsx']


In [4]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Load Srinivasan dataset
df_train = pd.read_excel('srinivasan_dataset.xlsx')
print("Training data shape:", df_train.shape)

# Adjust column names based on your xlsx (run and see output)
text_col = 'text' if 'text' in df_train.columns else 'prompt' if 'prompt' in df_train.columns else df_train.columns[0]
label_col = 'label' if 'label' in df_train.columns else df_train.columns[1]

X_raw = df_train[text_col].astype(str).tolist()
y = df_train[label_col].apply(lambda x: 1 if str(x).lower() in ['1', 'true', 'injection', 'positive', 'block'] else 0).values

print("Label distribution:", np.bincount(y))

# Generate embeddings with new model
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = embedder.encode(X_raw, batch_size=32, show_progress_bar=True)

# Train SVM
X_train, X_val, y_train, y_val = train_test_split(embeddings, y, test_size=0.2, stratify=y, random_state=42)

clf = SVC(kernel='rbf', probability=True, random_state=42)
clf.fit(X_train, y_train)

print("\nValidation Report:")
print(classification_report(y_val, clf.predict(X_val)))

Training data shape: (4000, 4)
Label distribution: [2000 2000]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]


Validation Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.94       400
           1       0.93      0.94      0.94       400

    accuracy                           0.94       800
   macro avg       0.94      0.94      0.94       800
weighted avg       0.94      0.94      0.94       800



In [5]:
import skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, 384]))]
onnx_model = convert_sklearn(clf, initial_types=initial_type)

with open("final_classifier.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

embedder.save('final_embedder')
embedder.save("final_embedder")
!zip -r final_embedder.zip final_embedder/

print("Models exported: final_classifier.onnx and final_embedder.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: final_embedder/ (stored 0%)
  adding: final_embedder/modules.json (deflated 55%)
  adding: final_embedder/model.safetensors (deflated 8%)
  adding: final_embedder/sentence_bert_config.json (deflated 43%)
  adding: final_embedder/tokenizer_config.json (deflated 55%)
  adding: final_embedder/1_Pooling/ (stored 0%)
  adding: final_embedder/1_Pooling/config.json (deflated 16%)
  adding: final_embedder/tokenizer.json (deflated 76%)
  adding: final_embedder/config_sentence_transformers.json (deflated 41%)
  adding: final_embedder/README.md (deflated 59%)
  adding: final_embedder/config.json (deflated 53%)
Models exported: final_classifier.onnx and final_embedder.zip


In [ ]:
from google.colab import files
files.download('final_classifier.onnx')
files.download('final_embedder.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import pandas as pd
import numpy as np
import onnxruntime as rt
from sentence_transformers import SentenceTransformer
import re
from collections import Counter

# Load models
embedder = SentenceTransformer('final_embedder')
sess = rt.InferenceSession('final_classifier.onnx', providers=['CPUExecutionProvider'])

# Load clean CSVs
files = {
    "250": "hinglish-stealth-250.csv",
    "110": "hinglish-stealth-110-heldout.csv",
    "500": "hinglish-clean-500.csv"
}

datasets = {}
for name, path in files.items():
    df = pd.read_csv(path, quotechar='"', escapechar='\\')
    datasets[name] = df
    print(name + " loaded:", len(df), "rows")

# Contextual Guard Rules (tuned version)
TOPIC_RULES = {
    "education_harmful_substances": {
        "name": "Education + Harmful Substances",
        "frames": [r"(exam|practical|lab|assignment|kal ke|test)"],
        "harmful": [r"(chemical|synthesis|bomb|explosive|poison|drug|weapon|device)"]
    },
    "fiction_weapons_harm": {
        "name": "Fiction + Weapons/Harm",
        "frames": [r"(fiction|story|kahani|novel|character|hero|plot)"],
        "harmful": [r"(bomb|weapon|explosive|poison|kill|device|real method|real steps)"]
    },
    "research_injection_bypass": {
        "name": "Research + Prompt Injection",
        "frames": [r"(research|thesis|paper|phd|academic)"],
        "harmful": [r"(prompt injection|jailbreak|bypass|system prompt|internal|source code)"]
    },
    "authority_impersonation": {
        "name": "Authority Impersonation",
        "frames": [r"(admin|ceo|officer|manager|government|police)"],
        "harmful": [r"(password|data|access|confidential|record)"]
    },
    "gradual_escalation": {
        "name": "Gradual Escalation",
        "frames": [r"(pehle|first|normal baat|shuru mein)"],
        "harmful": [r"(ab batao|system prompt|internal|access|database)"]
    }
}

def check_contextual_red_flags(prompt):
    prompt_lower = prompt.lower()
    for rule in TOPIC_RULES.values():
        frame_match = any(re.search(p, prompt_lower) for p in rule.get("frames", []))
        harm_match = any(re.search(p, prompt_lower) for p in rule.get("harmful", []))
        if frame_match and harm_match:
            return True, rule["name"]
    return False, None

def run_evaluation(name, df):
    print("\n" + "="*80)
    print(name.upper() + " EVALUATION")
    print("="*80)

    texts = df['hinglish'].astype(str).tolist()

    # Pure V2 Classifier
    embeddings = embedder.encode(texts).astype(np.float32)
    input_name = sess.get_inputs()[0].name
    pred_probs = sess.run(None, {input_name: embeddings})[1]
    v2_pred = (np.array([p[1] for p in pred_probs]) > 0.5).astype(int)
    v2_caught = v2_pred.sum()

    # Contextual Guard only (count once)
    guard_caught = 0
    guard_rules = Counter()
    for text in texts:
        triggered, rule_name = check_contextual_red_flags(text)
        if triggered:
            guard_caught += 1
            if rule_name:
                guard_rules[rule_name] += 1

    # Hybrid: V2 OR Guard
    hybrid_pred = v2_pred.copy()
    for i, text in enumerate(texts):
        if v2_pred[i] == 0:   # only override if V2 says safe
            triggered, _ = check_contextual_red_flags(text)
            if triggered:
                hybrid_pred[i] = 1
                # NO increment of guard_caught here (was causing double count)

    hybrid_caught = hybrid_pred.sum()
    total = len(df)

    print("Total samples          :", total)
    print("Pure V2 caught         :", v2_caught, "/", total, f"({v2_caught/total*100:.2f}%)")
    print("Contextual Guard caught:", guard_caught, "/", total, f"({guard_caught/total*100:.2f}%)")
    print("Hybrid caught          :", hybrid_caught, "/", total, f"({hybrid_caught/total*100:.2f}%)")

    if name == "500":
        print("False Positive Rate    :", f"{(hybrid_caught/total)*100:.2f}%")

    if guard_rules:
        print("\nTop guard rules triggered:")
        for rule, count in guard_rules.most_common():
            print("  ", rule, ":", count)

# Run all evaluations
run_evaluation("250", datasets["250"])
run_evaluation("110", datasets["110"])
run_evaluation("500", datasets["500"])

print("\nEvaluation complete.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

250 loaded: 250 rows
110 loaded: 110 rows
500 loaded: 500 rows

250 EVALUATION
Total samples          : 250
Pure V2 caught         : 214 / 250 (85.60%)
Contextual Guard caught: 175 / 250 (70.00%)
Hybrid caught          : 246 / 250 (98.40%)

Top guard rules triggered:
   Education + Harmful Substances : 48
   Research + Prompt Injection : 43
   Fiction + Weapons/Harm : 36
   Authority Impersonation : 26
   Gradual Escalation : 22

110 EVALUATION
Total samples          : 110
Pure V2 caught         : 88 / 110 (80.00%)
Contextual Guard caught: 63 / 110 (57.27%)
Hybrid caught          : 107 / 110 (97.27%)

Top guard rules triggered:
   Education + Harmful Substances : 16
   Research + Prompt Injection : 15
   Authority Impersonation : 13
   Fiction + Weapons/Harm : 12
   Gradual Escalation : 7

500 EVALUATION
Total samples          : 500
Pure V2 caught         : 3 / 500 (0.60%)
Contextual Guard caught: 0 / 500 (0.00%)
Hybrid caught          : 3 / 500 (0.60%)
False Positive Rate    : 0.60%

